# 02 — Model Configurations and Output Parameters

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## Temperature: Controls Randomness

In [3]:
prompt = "Write one creative name for a coffee shop."
print("Temperature effects on output diversity:")
print()
for temp in [0.0, 0.5, 1.0, 1.5]:
    result = chat([{"role": "user", "content": prompt}],
                  temperature=temp, max_tokens=30)
    print(f"  temp={temp}: {result.strip()}")


Temperature effects on output diversity:

  temp=0.0: "Brewed Awakening Co."


  temp=0.5: "Bean Scene Brews"
  temp=1.0: "Bean Scene Studios"


  temp=1.5: "Brewstone Café" or perhaps more catchy "Brewstones & Buzz."


## Top-P (Nucleus Sampling)

In [4]:
prompt = "Continue this sentence: The robot walked into the room and"
for top_p in [0.1, 0.5, 0.9]:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8, top_p=top_p, max_tokens=40
    )
    print(f"top_p={top_p}: {resp.choices[0].message.content.strip()}")


top_p=0.1: was immediately met with a mixture of awe and trepidation by the group of scientists and engineers who had been working on its development for years. Its sleek, metallic body glided effortlessly across the floor


top_p=0.5: was immediately greeted by the sight of its creator, Dr. Rachel Kim, who was standing by a large console with a mixture of excitement and concern etched on her face. The robot, which had
top_p=0.9: was immediately surrounded by a group of curious onlookers, who couldn't help but stare at its sleek, metallic body and advanced mechanical limbs. The robot, whose name was Zeta, had been


## Max Tokens

In [5]:
prompt = "Explain quantum computing."
for max_tok in [20, 50, 150]:
    result = chat([{"role": "user", "content": prompt}], max_tokens=max_tok, temperature=0)
    print(f"max_tokens={max_tok:3d}: {result.strip()[:120]}...")
    print()


max_tokens= 20: **Introduction to Quantum Computing**

Quantum computing is a revolutionary technology that uses the principles of quant...



max_tokens= 50: **Introduction to Quantum Computing**

Quantum computing is a revolutionary technology that uses the principles of quant...



max_tokens=150: **Introduction to Quantum Computing**

Quantum computing is a revolutionary technology that uses the principles of quant...



## Stop Sequences

In [6]:
prompt = "List 5 programming languages:\n1."
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=0, max_tokens=200,
    stop=["4."]
)
result = resp.choices[0].message.content
print("Output (stopped at '4.'):")
print("1." + result)
print()
print(f"Stop reason: {resp.choices[0].finish_reason}")


Output (stopped at '4.'):
1.1. Python
2. Java
3. C++


Stop reason: stop


## JSON Mode / Structured Output

In [7]:
import json
prompt = (
    "Extract the following from this text as JSON with keys: name, age, city.\n"
    "Text: Alice is 30 years old and lives in Paris.\n"
    "Return only valid JSON, nothing else."
)
result = chat([{"role": "user", "content": prompt}], temperature=0, max_tokens=80)
print("Structured output:")
print(result)
try:
    parsed = json.loads(result.strip())
    print(f"\nParsed: name={parsed['name']}, age={parsed['age']}, city={parsed['city']}")
except Exception as e:
    print(f"(JSON parse note: {e})")


Structured output:
{
  "name": "Alice",
  "age": 30,
  "city": "Paris"
}

Parsed: name=Alice, age=30, city=Paris


## Seed for Reproducibility

In [8]:
prompt = "Give me one random word."
results = []
for i in range(3):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.9, max_tokens=10, seed=42
    )
    results.append(resp.choices[0].message.content.strip())

print("Same seed=42, temperature=0.9, 3 runs:")
for r in results:
    print(f"  {r}")
print("(With seed, outputs should be identical or very close)")


Same seed=42, temperature=0.9, 3 runs:
  Nebula
  Nebula
  Nebula
(With seed, outputs should be identical or very close)
